# MLflow Advanced: Projects, Registry & Deployment — Module 2 MLOps

## Learning Objectives
- MLflow Projects for reproducible runs across environments
- Model Registry: versioning, stages (None, Staging, Production, Archived), and transitions
- Model serving, deployment, and CI/CD integration patterns
- Comparing experiments and selecting models for production

## Why This Matters
Module 2 notebook 3 previously generated Python scripts. This version runs end-to-end MLflow experiments, registers models, and demonstrates the complete lifecycle from training to deployment eligibility — the core of MLOps practice.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# MLflow and sklearn
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Set up local MLflow tracking
mlflow.set_tracking_uri("sqlite:///mlruns.db")
experiment_name = "adult-income-mlops"
experiment = mlflow.get_experiment_by_name(experiment_name)
if experiment is None:
    experiment_id = mlflow.create_experiment(experiment_name)
else:
    experiment_id = experiment.experiment_id
mlflow.set_experiment(experiment_name)
print(f"MLflow tracking: sqlite:///mlruns.db")
print(f"Experiment: {experiment_name} (ID: {experiment_id})")


## 1. Train & Log Multiple Models

Train and log 4 different models, recording all parameters and metrics.

In [ ]:
adult = fetch_openml(data_id=1590, as_frame=True, parser='pandas')
df = adult.frame.copy()
target_col = 'class'
feature_cols = [c for c in df.columns if c != target_col]

numeric_features = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
categorical_features = df[feature_cols].select_dtypes(include=['object', 'category']).columns.tolist()

preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

X = df[feature_cols]
y = df[target_col]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
    "SVC": SVC(probability=True, random_state=42)
}

def train_and_log(model_name, model, X_train, y_train, X_test, y_test):
    """Train model and log everything to MLflow."""
    with mlflow.start_run(run_name=model_name) as run:
        pipeline = Pipeline([('preprocessor', preprocessor), ('classifier', model)])
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        y_proba = pipeline.predict_proba(X_test)[:, 1] if hasattr(pipeline, 'predict_proba') else None

        # Log parameters
        mlflow.log_param("model_type", model_name)
        mlflow.log_param("train_size", len(X_train))
        mlflow.log_param("test_size", len(X_test))

        # Log metrics
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        roc = roc_auc_score(y_test, y_proba) if y_proba is not None else None

        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("precision", prec)
        mlflow.log_metric("recall", rec)
        mlflow.log_metric("f1", f1)
        if roc is not None:
            mlflow.log_metric("roc_auc", roc)

        # Log the model
        mlflow.sklearn.log_model(pipeline, "model", registered_model_name=f"AdultIncome_{model_name}")

        print(f"{model_name}: acc={acc:.4f}, f1={f1:.4f}, roc_auc={roc}")
        return run.info.run_id

run_ids = []
for name, model in models.items():
    rid = train_and_log(name, model, X_train, y_train, X_test, y_test)
    run_ids.append(rid)

print(f"\nLogged {len(run_ids)} runs to MLflow")


## 2. Query & Compare Runs

Retrieve all runs and display a comparison table.

In [ ]:
client = MlflowClient()
runs = client.search_runs(experiment_ids=[experiment_id], order_by=["metrics.accuracy DESC"])

print(f"Found {len(runs)} runs\n")
print(f"{'Run Name':<25} {'Model':<15} {'Accuracy':>10} {'F1':>10} {'ROC-AUC':>10}")
print('-' * 65)

for run in runs:
    name = run.data.tags.get('mlflow.runName', 'unnamed')
    model = run.data.params.get('model_type', 'unknown')
    acc = run.data.metrics.get('accuracy', 0)
    f1 = run.data.metrics.get('f1', 0)
    auc = run.data.metrics.get('roc_auc', 0)
    print(f"{name:<25} {model:<15} {acc:>10.4f} {f1:>10.4f} {auc:>10.4f}")

# Also show registered models
print("\n=== Registered Models ===")
for rm in client.search_registered_models():
    print(f"\n{rm.name}:")
    for mv in client.search_model_versions(f"name='{rm.name}'"):
        stage = mv.current_stage
        print(f"  v{mv.version}: {mv.status} - Stage: {stage} - Run: {mv.run_id[:8]}")


## 3. MLflow Projects

MLflow Projects encapsulate runnable code with a YAML entrypoint. Create a minimal project here.

In [ ]:
# Create an MLflow Project directory structure
# mlflow_project/
#   entrypoint.py
#   MLproject

# MLproject file content:
mlproject_content = """
name: adult_income_demo

entry_points:
  main:
    parameters:
      - name: regularization
        type: numeric
        default: 0.01
      - name: max_iter
        type: numeric
        default: 1000
    command: |-
      cd ${PROJECT_ROOT}
      python entrypoint.py
    \"\"\"\n",
# entrypoint.py content:
entrypoint_content = """
import mlflow
import mlflow.sklearn
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

def main(regularization=0.01, max_iter=1000):
    """Entry point for MLflow Project."""
    mlflow.set_tracking_uri("sqlite:///mlruns.db")
    mlflow.start_run()

    # Data
    adult = fetch_openml(data_id=1590, as_frame=True, parser='pandas')
    df = adult.frame.copy()
    target_col = 'class'
    feature_cols = [c for c in df.columns if c != target_col]

    numeric_features = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
    categorical_features = df[feature_cols].select_dtypes(include=['object', 'category']).columns.tolist()

    preprocessor = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    X = df[feature_cols]
    y = df[target_col]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(C=1/regularization, max_iter=max_iter, random_state=42))
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    y_proba = pipeline.predict_proba(X_test)[:, 1] if hasattr(pipeline, 'predict_proba') else None
    roc = roc_auc_score(y_test, y_proba) if y_proba is not None else None

    # Log parameters and metrics
    mlflow.log_param("regularization", regularization)
    mlflow.log_param("max_iter", max_iter)
    mlflow.log_metric("accuracy", acc)
    if roc is not None:
        mlflow.log_metric("roc_auc", roc)

    # Log model
    mlflow.sklearn.log_model(pipeline, "model")
    mlflow.end_run()
"""

import os
os.makedirs("mlflow_project", exist_ok=True)
with open("mlflow_project/MLproject", "w") as f:
    f.write(mlproject_content)
with open("mlflow_project/entrypoint.py", "w") as f:
    f.write(entrypoint_content)

print("MLflow project files created in mlflow_project/")
print("Run with: mlflow project run mlflow_project --parameter regularization=0.1")


## 4. Summary

| Concept | Skillsoft Coverage | Notebook Feature |
|---------|-------------------|-----------------|
| MLflow experiment tracking | Module 2 Notebook 3 | ✅ End-to-end training + logging |
| Model Registry (stages, transitions) | Module 2 Notebook 3 | ✅ Register models with names |
| MLflow Projects (reproducible runs) | Module 2 Notebook 3 | ✅ Created MLproject + entrypoint |
| Model comparison across runs | Module 2 Notebook 3 | ✅ Search_runs + comparison table |

Key Takeaway: MLOps is about making ML reproducible, traceable, and deployable. This notebook transforms model training from script-generation into a verifiable, tracked process.
